# Frengression versus Frugal Flows

This notebook runs the **same implementation** as [`compare_frugal_flows.py`](compare_frugal_flows.py). It compares the full interventional outcome distributions on one known binary-treatment, scalar-outcome simulation. The optional two-outcome study compares a joint Frengression fit to separate one-outcome Frengression fits and, if enabled, two separate scalar Frugal Flows fits. Combining two fitted outcome margins by independent sampling is **not** a multivariate Frugal Flows model.

**No results are pre-filled.** Select a Python 3.11+ kernel for this repository in VS Code, preferably a dedicated project environment, and run cells in order. The setup cell installs missing packages and Frugal Flows' documented FlowJAX version **into that kernel**; initial installation and JAX compilation can take time. Restart the kernel after a dependency change. **Run All** performs a wiring check and a moderate one-seed pilot by default. The optional diagnostic sweep is run separately after checking the pilot. After that, set `RUN_FULL = True` to run the five-seed benchmark; the diagnostic and full runs can take hours.


In [ ]:
from pathlib import Path
import importlib.metadata as metadata
import importlib.util
import subprocess
import sys

# Works when VS Code uses either the repository root or examples/ as cwd.
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "pyproject.toml").is_file()
             and (p / "examples" / "compare_frugal_flows.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("Open the frengression_pkg checkout in VS Code and use its notebook kernel.")

# Pinned to a Frugal Flows revision with sample_do and flexible_continuous.
FRUGAL_FLOWS_COMMIT = "26bc25cfa0c262e8a34af61841b202de79ffae0c"
FLOWJAX_VERSION = "19.1.0"  # From frugal-flows/environment.yaml.

def install(*args):
    subprocess.check_call([sys.executable, "-m", "pip", "install", *map(str, args)])

# This repository has a src/ layout. Put the checked-out code on the live
# notebook kernel's path even if pip's editable-install metadata is stale.
sys.path.insert(0, str(ROOT / "src"))
if any(importlib.util.find_spec(name) is None for name in ("torch", "engression")):
    install("-e", ROOT)
from frengression import Frengression  # Fail here rather than during the fit.

# Frugal Flows calls fit_to_data(data=...), which older FlowJAX versions do
# not accept. Install the version in Frugal Flows' own environment file.
try:
    installed_flowjax = metadata.version("flowjax")
except metadata.PackageNotFoundError:
    installed_flowjax = None
if installed_flowjax != FLOWJAX_VERSION:
    print(f"FlowJAX {installed_flowjax!r} -> {FLOWJAX_VERSION}; installing compatible version")
    install("flowjax==" + FLOWJAX_VERSION, "lineax==0.1.0", "paramax>=0.0.5")
    importlib.invalidate_caches()

if importlib.util.find_spec("frugal_flows") is None:
    install("git+https://github.com/llaurabatt/frugal-flows.git@" + FRUGAL_FLOWS_COMMIT)
missing = [name for name in ("scipy", "pandas", "matplotlib")
           if importlib.util.find_spec(name) is None]
if missing:
    install(*missing)
from frugal_flows.benchmarking import FrugalFlowModel  # Check the API up front.
from flowjax.train import fit_to_data
import inspect
if "data" not in inspect.signature(fit_to_data).parameters:
    raise RuntimeError("Old FlowJAX remains loaded. Restart the notebook kernel and rerun from the first cell.")

sys.path.insert(0, str(ROOT / "examples"))
from compare_frugal_flows import run, simulate
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
print("Using repository:", ROOT)
print("Python kernel:", sys.executable)


## Check the simulation

Treatment assignment depends on three covariates. Both methods see identical observational training data. Separate draws from the known data-generating process provide the intervention truth. The two outcomes share covariates and noise, so their **joint** distribution contains information that separate scalar margins cannot represent.


In [ ]:
from scipy.stats import pearsonr

x_obs, z_obs, y_obs = simulate(1000, seed=2026, dim=2)
_, _, y0 = simulate(3000, seed=2027, intervention=0, dim=2)
_, _, y1 = simulate(3000, seed=2028, intervention=1, dim=2)
fig, axes = plt.subplots(1, 2, figsize=(10, 3.6))
axes[0].hist(y0[:, 0], bins=45, density=True, alpha=0.55, label="do(0)")
axes[0].hist(y1[:, 0], bins=45, density=True, alpha=0.55, label="do(1)")
axes[0].set(xlabel="First outcome", ylabel="Density", title="Interventional margins")
axes[0].legend()
axes[1].scatter(y1[:500, 0], y1[:500, 1], s=7, alpha=0.35)
axes[1].set(xlabel="First outcome", ylabel="Second outcome",
            title=f"Joint do(1): correlation {pearsonr(y1[:, 0], y1[:, 1]).statistic:.2f}")
fig.tight_layout()
plt.show()


## Experiment settings

The smoke run (`n=200`, 20 Frengression updates by default) checks **only that the code runs**. Increasing its update count alone still leaves a small sample and one seed; do not compare its ATE errors. The pilot (`n=1000`, 1,000 Frengression updates) reports signed ATE estimates and arm means; the population ATE is exactly $1.5$. Re-run this configuration cell after editing a number and inspect the printed settings before fitting. Set `RUN_DIAGNOSTICS = True` and run the diagnostic cell **after** the pilot to separate the effect of more updates from the effect of more data. `RUN_FULL = True` enables the five-seed benchmark. `RUN_MULTIVARIATE` adds joint and separate-outcome Frengression fits in the full run. Set `RUN_FF_INDEPENDENT = True` to add two scalar Frugal Flows fits; these form an independent product, not a joint Frugal Flows model. Results go to `examples/benchmark_outputs/`.


In [ ]:
from argparse import Namespace

RUN_PILOT = True           # One seed with substantially longer fits.
RUN_DIAGNOSTICS = False    # Set True after pilot; run the diagnostic cell below.
RUN_FULL = False           # Set True after inspecting the pilot.
RUN_MULTIVARIATE = True  # Set False to run only the shared scalar comparison.
RUN_FF_INDEPENDENT = False  # True adds two scalar Frugal Flows fits per seed.
OUTPUT_DIR = ROOT / "examples" / "benchmark_outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

smoke = Namespace(n=200, repeats=1, seed=2026, mc=100, truth_mc=1000,
                  fr_iters=20, flow_epochs=20, marginal_epochs=20,
                  methods=["frengression", "frugal_flows"], multivariate=False,
                  output=str(OUTPUT_DIR / "smoke.csv"))
pilot = Namespace(n=1000, repeats=1, seed=2026, mc=1000, truth_mc=20000,
                  fr_iters=1000, flow_epochs=400, marginal_epochs=100,
                  methods=["frengression", "frugal_flows"], multivariate=False,
                  output=str(OUTPUT_DIR / "pilot.csv"))
full = Namespace(n=2000, repeats=5, seed=2026, mc=1000, truth_mc=50000,
                 fr_iters=2000, flow_epochs=2000, marginal_epochs=400,
                 methods=["frengression", "frugal_flows"],
                 multivariate=RUN_MULTIVARIATE,
                 multivariate_ff=RUN_MULTIVARIATE and RUN_FF_INDEPENDENT,
                 output=str(OUTPUT_DIR / "comparison_results.csv"))
for label, settings in (("smoke", smoke), ("pilot", pilot), ("full", full)):
    print(f"{label}: n={settings.n}, repeats={settings.repeats}, "
          f"fr_iters={settings.fr_iters}, flow_epochs={settings.flow_epochs}, "
          f"marginal_epochs={settings.marginal_epochs}, mc={settings.mc}")
print("Pilot:", RUN_PILOT, "| Diagnostic sweep:", RUN_DIAGNOSTICS,
      "| Full experiment:", RUN_FULL)


## 1. Wiring check

This actually trains each scalar model briefly. Small training budgets mean these numbers **must not** be used in the paper. If installation, model API, or memory fails, send the full error message and Python/package versions for debugging.


In [ ]:
run(smoke)
smoke_results = pd.read_csv(smoke.output)
assert set(smoke_results.method) == {"frengression", "frugal_flows"}
print(f"Wiring check passed (n={smoke.n}, Frengression updates={smoke.fr_iters}). "
      "Ignore smoke accuracy even if you increased its update count.")


## 2. One-seed pilot

This uses 1,000 observations and 1,000 Frengression optimisation steps by default, versus 400 Frugal Flows epochs and 100 marginal-flow epochs. These counts are **not comparable units of compute**. Inspect each fitted arm mean against the true arm mean and the signed ATE; the observational arm means are confounded and are **not** causal targets. One seed is a diagnostic, not a publication comparison. If the pilot remains far from the true ATE, use the optional sweep below before launching the five-seed run.


In [ ]:
if RUN_PILOT:
    print(f"Running pilot: n={pilot.n}, Frengression updates={pilot.fr_iters}, "
          f"Frugal Flows epochs={pilot.flow_epochs}, MC draws={pilot.mc}")
    pilot_signature = (pilot.n, pilot.fr_iters, pilot.seed, pilot.mc, pilot.truth_mc)
    run(pilot)
    pilot_results = pd.read_csv(pilot.output)
    display(pilot_results[["method", "seconds", "true_ate", "estimated_ate", "ate_error",
                           "true_mean0", "estimated_mean0", "true_mean1", "estimated_mean1",
                           "wasserstein_mean", "energy_mean", "quantile_mae"]])
    x_observed, _, y_observed = simulate(pilot.n, pilot.seed)
    print("Observed arm means (confounded; not intervention truth):",
          {arm: float(y_observed[x_observed[:, 0] == arm, 0].mean())
           for arm in (0, 1)})
else:
    print("Skipped pilot. Set RUN_PILOT = True in the configuration cell to run it.")


## 2a. Optional diagnosis: training budget or sample size?

After running the pilot, set `RUN_DIAGNOSTICS = True` in the configuration cell, re-run that cell, and run **only the next cell**. It keeps the pilot's `n=1000, fr_iters=1000` Frengression row, then fits Frengression with `(n=1000, fr_iters=2000)` and `(n=2000, fr_iters=2000)`. The first comparison probes insufficient optimisation; the second probes sample size at a fixed training budget. Only the Frengression fits are repeated. These one-seed checks can take time and do not establish superiority. Check both arm means and distributional error; if neither change helps, investigate fit and architecture rather than simply increasing iterations. Results are saved to separate CSV files.


In [ ]:
if RUN_DIAGNOSTICS:
    if not RUN_PILOT or "pilot_results" not in globals():
        raise RuntimeError("Run the pilot cell in this kernel before the diagnostic sweep.")
    if pilot.n != 1000 or pilot.fr_iters != 1000:
        raise RuntimeError("This sweep expects the default pilot n=1000, fr_iters=1000.")
    if "pilot_signature" not in globals() or pilot_signature != (pilot.n, pilot.fr_iters, pilot.seed, pilot.mc, pilot.truth_mc):
        raise RuntimeError("Re-run the pilot cell after changing its configuration.")
    diagnostic_rows = []
    baseline = pilot_results.loc[pilot_results.method == "frengression"].copy()
    if len(baseline) != 1:
        raise RuntimeError("Expected exactly one Frengression pilot row.")
    baseline.insert(0, "n", pilot.n)
    baseline.insert(1, "fr_iters", pilot.fr_iters)
    diagnostic_rows.append(baseline)
    for n_samples, n_steps in ((1000, 2000), (2000, 2000)):
        settings = Namespace(**vars(pilot))
        settings.n = n_samples
        settings.fr_iters = n_steps
        settings.methods = ["frengression"]
        settings.output = str(OUTPUT_DIR / f"diagnostic_fr_n{n_samples}_steps{n_steps}.csv")
        print(f"Diagnostic Frengression: n={settings.n}, updates={settings.fr_iters}, "
              f"seed={settings.seed}, MC draws={settings.mc}")
        run(settings)
        row = pd.read_csv(settings.output)
        row.insert(0, "n", n_samples)
        row.insert(1, "fr_iters", n_steps)
        diagnostic_rows.append(row)
    diagnostic_results = pd.concat(diagnostic_rows, ignore_index=True)
    display(diagnostic_results[[
        "n", "fr_iters", "true_ate", "estimated_ate", "ate_error",
        "true_mean0", "estimated_mean0", "true_mean1", "estimated_mean1",
        "wasserstein_mean", "energy_mean", "seconds"
    ]])
else:
    print("Optional diagnostic skipped. Set RUN_DIAGNOSTICS = True, "
          "re-run the configuration cell, then run this cell after the pilot.")


## 3. Publication candidate run

After checking the pilot and, if needed, the optional diagnosis, set `RUN_FULL = True` in the configuration cell, rerun that cell, then run this cell and the summaries below. A benchmark CSV is written after the fits finish; if a run fails partway through, the script currently does not checkpoint intermediate seeds. Choose training budgets using diagnostics, then report every method and metric across seeds rather than selecting a favourable run.


In [ ]:
if RUN_FULL:
    run(full)
    results = pd.read_csv(full.output)
    display(results)
else:
    print("Skipped full run. Set RUN_FULL = True in the configuration cell, then rerun it and this cell.")


## 4. Compare across seeds

Lower is better for every error column. The scalar comparison is **head-to-head**. Fit time is diagnostic only: Frugal Flows times marginal CDF plus its flow, whereas Frengression times its `f,h` fit and omits `g`. The two-outcome table compares joint Frengression with products of separately fitted scalar margins. With `RUN_FF_INDEPENDENT = True`, it includes the product of two Frugal Flows fits. Check marginal Wasserstein error alongside joint energy and cross-covariance error: a joint difference could otherwise reflect poorer scalar margins. This does not compare two joint-outcome methods.


In [ ]:
if RUN_FULL and Path(full.output).is_file():
    scalar = results.loc[results.setting == "binary_scalar"]
    scalar_metrics = ["ate_error", "wasserstein_mean", "energy_mean", "quantile_mae"]
    display(scalar.groupby("method")[scalar_metrics].agg(["mean", "std"]).round(4))
    if RUN_MULTIVARIATE:
        joint = results.loc[results.setting == "binary_two_outcomes"]
        joint_metrics = ["joint_energy_mean", "marginal_wasserstein_mean",
                         "mean_error", "covariance_error", "cross_covariance_error"]
        display(joint.groupby("method")[joint_metrics].agg(["mean", "std"]).round(4))
else:
    print("Run the full experiment first; the one-seed smoke numbers are not paper results.")


## Interpretation and reproducibility

A useful strength claim would require consistently good **distributional** accuracy in the shared scalar task and, separately, smaller joint-distribution/covariance error for the joint fit while keeping marginal errors acceptable. Report all metrics, variation across seeds, the data-generating process, training budgets, hardware, and code versions, even if a competitor wins a metric. This is one simulation setting; add a distinct setting and assess sensitivity before claiming general superiority.

The procedure estimates interventional distributions under its stated simulation assumptions. Do not interpret the displayed seed percentiles as confidence intervals or use the two-outcome test as proof of correct longitudinal joint trajectories.


In [ ]:
import importlib.metadata as metadata

for package in ("frengression", "frugal-flows", "torch", "jax", "flowjax", "numpy", "scipy"):
    try:
        print(f"{package}: {metadata.version(package)}")
    except metadata.PackageNotFoundError:
        print(f"{package}: version unavailable")
try:
    revision = subprocess.check_output(["git", "-C", str(ROOT), "rev-parse", "HEAD"], text=True).strip()
    print("frengression_pkg commit:", revision)
except (subprocess.CalledProcessError, FileNotFoundError):
    pass
print("Frugal Flows install target:", FRUGAL_FLOWS_COMMIT)
if RUN_FULL and Path(full.output).is_file():
    print("Share this CSV for interpretation:", full.output)
